# Descriptors and Properties

Control what happens when attributes are accessed. This is how `@property` and many frameworks work.

## The @property Decorator

In [ ]:
class Circle:
    def __init__(self, radius):
        self._radius = radius
    
    @property
    def radius(self):
        """Getter - called when accessing circle.radius"""
        return self._radius
    
    @radius.setter
    def radius(self, value):
        """Setter - called when assigning circle.radius = x"""
        if value < 0:
            raise ValueError("Radius cannot be negative")
        self._radius = value
    
    @property
    def area(self):
        """Read-only computed property"""
        return 3.14159 * self._radius ** 2

c = Circle(5)
print(f"Radius: {c.radius}")
print(f"Area: {c.area}")

c.radius = 10
print(f"New area: {c.area}")

try:
    c.radius = -5
except ValueError as e:
    print(f"Error: {e}")

In [ ]:
# Property with deleter
class User:
    def __init__(self, name):
        self._name = name
    
    @property
    def name(self):
        return self._name
    
    @name.setter
    def name(self, value):
        self._name = value
    
    @name.deleter
    def name(self):
        print("Deleting name...")
        self._name = "Anonymous"

u = User("Alice")
print(f"Name: {u.name}")

del u.name
print(f"After delete: {u.name}")

## Common Property Patterns

In [ ]:
# Pattern 1: Lazy computed property with caching
class DataProcessor:
    def __init__(self, data):
        self._data = data
        self._result = None
    
    @property
    def result(self):
        if self._result is None:
            print("Computing result (expensive!)...")
            self._result = sum(x**2 for x in self._data)
        return self._result

proc = DataProcessor([1, 2, 3, 4, 5])
print(f"First access: {proc.result}")   # Computes
print(f"Second access: {proc.result}")  # Uses cache

In [ ]:
# Pattern 2: Validation on setter
class Config:
    def __init__(self):
        self._port = 8080
    
    @property
    def port(self):
        return self._port
    
    @port.setter
    def port(self, value):
        if not isinstance(value, int):
            raise TypeError("Port must be an integer")
        if not 0 <= value <= 65535:
            raise ValueError("Port must be 0-65535")
        self._port = value

config = Config()
config.port = 3000
print(f"Port: {config.port}")

try:
    config.port = "invalid"
except TypeError as e:
    print(f"Error: {e}")

In [ ]:
# Pattern 3: Computed property from other attributes
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height
    
    @property
    def area(self):
        return self.width * self.height
    
    @property
    def perimeter(self):
        return 2 * (self.width + self.height)

r = Rectangle(4, 5)
print(f"Area: {r.area}")
print(f"Perimeter: {r.perimeter}")

r.width = 10
print(f"New area: {r.area}")

## Descriptors - How Properties Work

In [ ]:
# A descriptor is a class with __get__, __set__, or __delete__
class Validated:
    """Descriptor that validates values."""
    
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f'__{name}'
    
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.storage_name, None)
    
    def __set__(self, obj, value):
        self.validate(value)
        setattr(obj, self.storage_name, value)
    
    def validate(self, value):
        pass  # Override in subclass

class PositiveNumber(Validated):
    def validate(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f"{self.name} must be a number")
        if value <= 0:
            raise ValueError(f"{self.name} must be positive")

class Product:
    price = PositiveNumber()
    quantity = PositiveNumber()
    
    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity

p = Product("Widget", 9.99, 100)
print(f"Product: {p.name}, ${p.price} x {p.quantity}")

try:
    p.price = -5
except ValueError as e:
    print(f"Error: {e}")

## `__slots__` - Memory Optimization

In [ ]:
# Regular class uses __dict__ for attributes
class RegularPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

# __slots__ class uses fixed attributes (less memory)
class SlottedPoint:
    __slots__ = ['x', 'y']
    
    def __init__(self, x, y):
        self.x = x
        self.y = y

regular = RegularPoint(3, 4)
slotted = SlottedPoint(3, 4)

print(f"Regular has __dict__: {hasattr(regular, '__dict__')}")
print(f"Slotted has __dict__: {hasattr(slotted, '__dict__')}")

# Can add arbitrary attributes to regular
regular.z = 5
print(f"Added z to regular: {regular.z}")

# Cannot add to slotted
try:
    slotted.z = 5
except AttributeError as e:
    print(f"Cannot add to slotted: {e}")

## `__getattr__` and `__getattribute__`

In [ ]:
class DynamicAttributes:
    def __init__(self):
        self.existing = "I exist"
    
    def __getattr__(self, name):
        """Called only when attribute not found normally."""
        return f"Dynamic value for '{name}'"

obj = DynamicAttributes()
print(f"existing: {obj.existing}")        # Normal access
print(f"missing: {obj.missing}")          # Calls __getattr__
print(f"anything: {obj.anything_at_all}") # Calls __getattr__

In [ ]:
# Practical use: Dictionary wrapper
class AttrDict:
    """Access dict keys as attributes."""
    
    def __init__(self, data):
        self._data = data
    
    def __getattr__(self, name):
        try:
            return self._data[name]
        except KeyError:
            raise AttributeError(f"No attribute '{name}'")
    
    def __setattr__(self, name, value):
        if name.startswith('_'):
            super().__setattr__(name, value)
        else:
            self._data[name] = value

config = AttrDict({"host": "localhost", "port": 8080})
print(f"Host: {config.host}")
print(f"Port: {config.port}")

config.debug = True
print(f"Debug: {config.debug}")

## AI Code Patterns

In [ ]:
# Pattern 1: Cached property (Python 3.8+)
from functools import cached_property

class DataAnalyzer:
    def __init__(self, data):
        self.data = data
    
    @cached_property
    def statistics(self):
        """Computed once, then cached."""
        print("Computing statistics...")
        return {
            'mean': sum(self.data) / len(self.data),
            'min': min(self.data),
            'max': max(self.data),
        }

analyzer = DataAnalyzer([1, 2, 3, 4, 5])
print(f"First: {analyzer.statistics}")   # Computes
print(f"Second: {analyzer.statistics}")  # Cached

In [ ]:
# Pattern 2: Type-checked attributes
class TypedProperty:
    def __init__(self, expected_type):
        self.expected_type = expected_type
    
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f'_{name}'
    
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.storage_name, None)
    
    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name} must be {self.expected_type.__name__}, "
                f"got {type(value).__name__}"
            )
        setattr(obj, self.storage_name, value)

class Person:
    name = TypedProperty(str)
    age = TypedProperty(int)
    
    def __init__(self, name, age):
        self.name = name
        self.age = age

p = Person("Alice", 30)
print(f"{p.name}, {p.age}")

try:
    p.age = "thirty"
except TypeError as e:
    print(f"Error: {e}")

In [ ]:
# Pattern 3: Property that clears cache when set
class CacheInvalidatingProperty:
    def __init__(self, cache_attrs):
        self.cache_attrs = cache_attrs
    
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f'_{name}'
    
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.storage_name)
    
    def __set__(self, obj, value):
        setattr(obj, self.storage_name, value)
        # Clear cached properties
        for attr in self.cache_attrs:
            if attr in obj.__dict__:
                del obj.__dict__[attr]

print("CacheInvalidatingProperty defined")

## Summary

| Feature | Purpose |
|---------|--------|
| `@property` | Make method look like attribute |
| `@x.setter` | Control attribute assignment |
| `@cached_property` | Compute once, cache result |
| Descriptor | Reusable attribute behavior |
| `__getattr__` | Handle missing attributes |
| `__slots__` | Memory-efficient fixed attributes |

## Next Up

Async/await - concurrent programming.

Continue to: [Async/Await](04-async-await.ipynb)